# Mnemonics LongMemEval — 50q + 500q

Hiçbir şey yüklemenize gerek yok. Dataset HuggingFace'ten otomatik inecek (~265 MB).  
**Runtime → Change runtime type → T4 GPU** seçin, sonra tüm hücreleri sırayla çalıştırın.

## 1) Repo + bağımlılıklar

In [ ]:
!git clone https://github.com/nakata-app/mnemonics.git /content/mnemonics
%cd /content/mnemonics
!git log --oneline -3

In [ ]:
!pip install -q -e . sentence-transformers numpy 2>&1 | tail -5
print('Install done')

## 2) Dataset indir (HuggingFace → local JSON)

In [ ]:
import os
DATA = '/content/longmemeval_s.json'
if not os.path.exists(DATA):
    print('Downloading dataset (~265 MB)...')
    !wget -q --show-progress -O {DATA} \
        'https://huggingface.co/datasets/xiaowu0162/LongMemEval/resolve/main/longmemeval_s.json'
else:
    print('Dataset zaten var, atlanıyor')
import os; print(f'Size: {os.path.getsize(DATA)/1e6:.1f} MB')

In [ ]:
# Eval script DATA path'ini patch'le
import re, pathlib
p = pathlib.Path('/content/mnemonics/benchmarks/longmemeval_eval.py')
src = p.read_text()
src = re.sub(r'DATA = Path\([^)]+\)', f'DATA = Path("{DATA}")', src)
# MemPalace baseline olmayacak, missing kontrolü zaten var
p.write_text(src)
!grep -n 'DATA = Path' {p}
print('Patch OK')

## 3) Smoke test — 5 soru, hızlı kontrol

In [ ]:
!cd /content/mnemonics && python benchmarks/longmemeval_eval.py \
    --n 5 --mode no_rerank \
    --augment-preferences --augment-assistant-facts --candidate-k 50 \
    --out /tmp/smoke.json && echo '=== SMOKE OK ==='

## 4) 50q ablasyon — baseline vs facts (aynı seed=42)

Aynı 50 soruyu iki kez: biri facts OFF, biri facts ON. Delta = net katkı.

In [ ]:
os.makedirs('/content/results', exist_ok=True)

# 4A) Baseline: facts OFF
print('=== 4A: 50q baseline (facts OFF) ===')
!cd /content/mnemonics && python benchmarks/longmemeval_eval.py \
    --n 50 --mode rerank \
    --augment-preferences --candidate-k 50 --seed 42 \
    --out /content/results/lme50_baseline.json \
    --per-q-out /content/results/lme50_baseline_perq.json

In [ ]:
# 4B) Facts ON
print('=== 4B: 50q facts ON ===')
!cd /content/mnemonics && python benchmarks/longmemeval_eval.py \
    --n 50 --mode rerank \
    --augment-preferences --augment-assistant-facts --candidate-k 50 --seed 42 \
    --out /content/results/lme50_facts.json \
    --per-q-out /content/results/lme50_facts_perq.json

In [ ]:
# 4C) Delta tablosu
import json
b = json.load(open('/content/results/lme50_baseline.json'))['mnemonics_rerank']
f = json.load(open('/content/results/lme50_facts.json'))['mnemonics_rerank']
print(f'{'Metrik':8} {'Baseline':>10} {'Facts':>10} {'Delta':>8}')
print('-' * 40)
for k in ('R@1', 'R@5', 'R@10'):
    delta = f[k] - b[k]
    arrow = '↑' if delta > 0 else ('↓' if delta < 0 else '=')
    print(f'{k:8} {b[k]:>10.3f} {f[k]:>10.3f} {delta:>+7.3f} {arrow}')

In [ ]:
# 4D) Kurtarılan vs kaybedilen sorular (R@10)
pb = {r['qid']: r for r in json.load(open('/content/results/lme50_baseline_perq.json'))}
pf = {r['qid']: r for r in json.load(open('/content/results/lme50_facts_perq.json'))}
rescued, broken = [], []
for qid in pb:
    if qid not in pf: continue
    if pf[qid]['hit@10'] and not pb[qid]['hit@10']:
        rescued.append((qid, pb[qid]['qtype'], pb[qid]['question'][:90]))
    elif pb[qid]['hit@10'] and not pf[qid]['hit@10']:
        broken.append((qid, pb[qid]['qtype'], pb[qid]['question'][:90]))

print(f'RESCUED — facts ile yakalandı ({len(rescued)} soru):')
for qid, t, q in rescued:
    print(f'  [{t}] {q}')
print(f'\nBROKEN — facts ile kaybedildi ({len(broken)} soru):')
for qid, t, q in broken:
    print(f'  [{t}] {q}')

## 5) 500q full eval — baseline + facts

T4 GPU'da ~90-120 dk her biri. İkisi sıralı, toplamda ~3-4 saat.  
Colab bu süre boyunca açık kalmalı (idle timeout'u engellemek için).

In [ ]:
# 5A) 500q baseline
print('=== 5A: 500q baseline (facts OFF) — uzun sürecek ===')
!cd /content/mnemonics && python benchmarks/longmemeval_eval.py \
    --n 500 --mode rerank \
    --augment-preferences --candidate-k 50 --seed 42 \
    --out /content/results/lme500_baseline.json \
    --per-q-out /content/results/lme500_baseline_perq.json

In [ ]:
# 5B) 500q facts ON
print('=== 5B: 500q facts ON ===')
!cd /content/mnemonics && python benchmarks/longmemeval_eval.py \
    --n 500 --mode rerank \
    --augment-preferences --augment-assistant-facts --candidate-k 50 --seed 42 \
    --out /content/results/lme500_facts.json \
    --per-q-out /content/results/lme500_facts_perq.json

In [ ]:
# 5C) 500q delta + type breakdown
b5 = json.load(open('/content/results/lme500_baseline.json'))['mnemonics_rerank']
f5 = json.load(open('/content/results/lme500_facts.json'))['mnemonics_rerank']
print('=== OVERALL ===')
for k in ('R@1', 'R@5', 'R@10'):
    print(f'{k:6}  baseline={b5[k]:.3f}  facts={f5[k]:.3f}  Δ={f5[k]-b5[k]:+.3f}')
print('\n=== BY TYPE ===')
for qt in sorted(b5['by_type']):
    bb = b5['by_type'][qt]
    ff = f5['by_type'].get(qt, {})
    d = ff.get('R@10', 0) - bb['R@10']
    print(f'{qt:25} n={bb["n"]:3}  base={bb["R@10"]:.3f}  facts={ff.get("R@10",0):.3f}  Δ={d:+.3f}')

In [ ]:
# 5D) Sonuçları Drive'a yedekle (Drive mount açıksa)
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    import shutil, os
    dest = '/content/drive/MyDrive/lme/results'
    os.makedirs(dest, exist_ok=True)
    for f in os.listdir('/content/results'):
        shutil.copy(f'/content/results/{f}', f'{dest}/{f}')
    print('Drive backup OK:', dest)
except Exception as e:
    print('Drive yok veya mount edilmedi, sonuçlar /content/results/ de:')
    !ls -lh /content/results/